# 064 — Tokenización y representación del lenguaje

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Tokenización:** por palabras (vocabulario enorme + `<UNK>`) y por caracteres (secuencias
larguísimas) son extremos fallidos; las **subpalabras** son el punto medio. **BPE** parte
de caracteres y fusiona iterativamente el par adyacente más frecuente hasta el tamaño de
vocabulario deseado; el texto nuevo se tokeniza aplicando las fusiones en orden. GPT usa
BPE sobre **bytes** (nunca hay `<UNK>`); **WordPiece** (BERT) marca fragmentos con `##`;
**SentencePiece** trata el espacio como símbolo (`▁`).

**Representación:** one-hot (sin similitud) → bolsa de palabras (conteos, sin orden) →
TF-IDF (conteos ponderados por rareza) → **embeddings densos**: tabla `|V|×d` aprendida
donde tokens de contextos similares acaban cerca.

**Consecuencias:** costo y ventana de los LLM se miden en tokens (~1.4–1.8 por palabra en
español); los idiomas subrepresentados se fragmentan más y pagan más.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Pares: `(h,u)=10, (u,g)=15, (g,_)=15, (p,u)=17, (u,n)=16, (n,_)=16,
(b,u)=4`. Máximo: `(p,u)=17` → **fusión 1: pu**. Recontando sobre el corpus fusionado:
`(pu,g)=5, (pu,n)=12, (u,g)=10, (u,n)=4, (g,_)=15, (n,_)=16, (h,u)=10, (b,u)=4`; nota que
`(u,n)` cae de 16 a 4 porque en `pun` la `u` ya está dentro de `pu`. Gana `(n,_)=16` →
**fusión 2: n_**. Resultado: `hug → h u g _`, `pug → pu g _`, `pun → pu n_`,
`bun → b u n_`.

**Ejercicio 2.** (a) `lowest_`: `l o w e s t _` → es → est → est_ → lo → low ⇒
`low + est_`. (b) `west_`: `w e s t _` → `w + est_` (las reglas lo, low no aplican).
(c) `lot_`: `l o t _` → lo ⇒ `lo + t + _` — `t` y `_` quedan sueltos porque ninguna regla
los fusiona en este contexto.

**Ejercicio 3.** Ambas frases producen el **mismo vector** `[1, 1, 1, 1, 1]`: BoW ignora
el orden, y quién muerde a quién desaparece. Lo resuelve una representación secuencial
(n-gramas capturan algo; embeddings contextuales de un transformer capturan la relación
completa).

**Ejercicio 4.** El conteo en código confirma `(p,u) = 17` como primer par a fusionar.


In [ ]:
result = run_lab("llm", seed=64)
assert result["kind"] == "llm"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 4 — conteo de pares BPE verificado con código
from collections import Counter

corpus = {("h", "u", "g", "_"): 10, ("p", "u", "g", "_"): 5,
          ("p", "u", "n", "_"): 12, ("b", "u", "n", "_"): 4}

def contar_pares(corpus):
    pares = Counter()
    for palabra, freq in corpus.items():
        for a, b in zip(palabra, palabra[1:]):
            pares[(a, b)] += freq
    return pares

def fusionar(corpus, par):
    nuevo = {}
    for palabra, freq in corpus.items():
        out, i = [], 0
        while i < len(palabra):
            if i < len(palabra) - 1 and (palabra[i], palabra[i + 1]) == par:
                out.append(palabra[i] + palabra[i + 1])
                i += 2
            else:
                out.append(palabra[i])
                i += 1
        nuevo[tuple(out)] = freq
    return nuevo

pares = contar_pares(corpus)
print("top:", pares.most_common(3))          # (p,u) = 17
corpus2 = fusionar(corpus, ("p", "u"))
print("tras fusion 1:", contar_pares(corpus2).most_common(3))  # (n,_) = 16


In [ ]:
# Ejercicio 3 — BoW verificado con código
vocab = ["el", "perro", "niño", "muerde", "al"]

def bow(frase):
    tokens = frase.split()
    return [tokens.count(v) for v in vocab]

a = bow("el perro muerde al niño")
b = bow("el niño muerde al perro")
print(a, b, "identicos:", a == b)  # BoW pierde el orden


## Reflexión

1. El laboratorio `llm` es un modelo didáctico determinista. En un LLM real, ¿por qué el
   mismo prompt puede costar distinto número de tokens en GPT y en BERT, y qué implica
   para comparar "longitudes de contexto" entre modelos?
2. Si tu producto atiende usuarios en español y guaraní con un tokenizador entrenado sobre
   todo en inglés, ¿quién paga más por mensaje y qué medirías para cuantificarlo?
3. ¿Qué se rompe aguas abajo (índices, cachés, embeddings) si actualizas el tokenizador de
   un sistema en producción sin reentrenar el modelo?
